[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/12_greedy.ipynb)

*DSA Interview Handbook · Notebook 12 of 14*

# Greedy & Intervals

Make the locally best choice and prove it never hurts — usually after sorting, often on intervals.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Greedy takes the best-looking step now and never looks back. It's only right when you can prove it.

**What interviewers really test here:** Justifying the greedy choice, interval sorting, and when greedy fails.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. What greedy means, and when it fails *(Basics)*
- [ ] 2. Proving a greedy: the exchange argument *(Core)*
- [ ] 3. Intervals: sort, then merge *(Core)*
- [ ] 4. Scheduling: sort by end time *(Core)*
- [ ] 5. Reach and running balance *(Core)*
- [ ] 6. Sweep lines and two passes *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **greedy algorithm** builds a solution step by step, at each step making the choice that looks **best at that moment** (the locally optimal choice), and never reconsidering earlier choices. It is correct only for problems where local optimality leads to global optimality.



An **interval** is a pair [start, end]. Interval problems (merging, scheduling, covering) are among the most common uses of greedy algorithms.

### When greedy works

#### 1. Greedy-choice property



A globally optimal solution can be reached by making a locally optimal choice first. There exists some optimal solution that agrees with the greedy choice.



#### 2. Optimal substructure



After making the greedy choice, what remains is a smaller instance of the same problem, whose optimal solution combines with the choice to give an optimal overall solution.



#### How to prove it




- **Exchange argument**: take any optimal solution that differs from greedy; swap in the greedy choice and show the result is no worse.
- **Greedy stays ahead**: show after each step greedy's partial solution is at least as good as any other's.
- **Counter-example search**: before proving, try small inputs to break the idea. If one breaks it, switch to DP.

### Classic greedy problems

| Problem | Greedy rule | Time |
| --- | --- | --- |
| Activity selection / max non-overlapping intervals | pick the interval that **ends earliest** | O(n log n) |
| Fractional knapsack | take items by highest value/weight ratio, splitting the last | O(n log n) |
| Huffman coding | repeatedly merge the two least frequent symbols | O(n log n) |
| Minimum spanning tree (Kruskal, Prim) | add the cheapest safe edge | O(E log V) |
| Dijkstra | finalise the closest unvisited vertex | O(E log V) |
| Job sequencing with deadlines | schedule highest profit first in the latest free slot | O(n log n) |
| Coin change (canonical coins like 1, 2, 5, 10) | take the largest coin that fits | O(amount / coin) |



**Greedy fails** for 0/1 knapsack and for coin change with arbitrary coins (e.g. coins 1, 3, 4 and amount 6) — those need DP.

### Interval theory

Two intervals [a, b] and [c, d] **overlap** when `a ≤ d and c ≤ b` (use &lt; if touching endpoints don't count). Their intersection is `[max(a, c), min(b, d)]` when non-empty.



| Goal | Sort by | Why |
| --- | --- | --- |
| Merge overlapping intervals | start | overlaps then become adjacent in order |
| Maximum non-overlapping set / minimum removals | end | finishing earliest leaves the most room (exchange argument) |
| Minimum points to stab all intervals | end | a point at the earliest end hits every interval that started before it |
| Maximum overlap (rooms, platforms) | events (start +1, end −1) | sweep line counts active intervals |
| Remove covered intervals | start asc, end desc | a covered interval has end ≤ the max end seen so far |

### Greedy vs dynamic programming

|   | Greedy | Dynamic programming |
| --- | --- | --- |
| Choices per step | one (the locally best) | all, keeps the best |
| Reconsiders? | never | effectively yes |
| Correctness | needs a proof (greedy-choice property) | follows from optimal substructure |
| Speed | usually O(n log n) or O(n) | usually O(n²) or O(n · W) |
| Memory | O(1) extra | table of states |
| Example where it's right | fractional knapsack | 0/1 knapsack |

### Advantages, disadvantages and applications

**👍 Pros**

##### Advantages

- Simple and fast
- Little memory
- Often the optimal method for scheduling and MST problems
- Good approximation when exact methods are too slow



**👎 Cons**

##### Disadvantages

- Not always correct — needs proof
- Hard to know whether it applies without experience
- Can give a poor answer silently





#### Applications




- CPU and meeting scheduling, room / platform allocation, calendar apps.
- Data compression (Huffman coding).
- Network design (minimum spanning trees), routing (Dijkstra).
- Making change at a cash register (canonical coin systems).
- Load balancing and caching policies; approximation algorithms for NP-hard problems (set cover).

---
# Part 2: Lessons, step by step

## Lesson 1: What greedy means, and when it fails

*Level: Basics*

A **greedy** algorithm makes the choice that looks best *right now* and never goes back. It's simple and fast — but only correct for some problems.



Coins `[1, 3, 4]`, amount 6. Greedy takes the biggest coin first: 4 + 1 + 1 = **3 coins**. The best answer is 3 + 3 = **2 coins**. Greedy failed, so this problem needs DP.



Before trusting a greedy idea, always try to **break it with a small example**. If you can't, look for a proof (next lesson).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def greedy_fail_lesson():
    coins, amount = [4, 3, 1], 6
    left, used = amount, []
    for c in coins:
        while left >= c:
            left -= c
            used.append(c)
    return len(used)


greedy_fail_lesson()

3

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Greedy takes the largest coin that fits: 4, 2 left
2. Greedy takes the largest coin that fits: 1, 1 left
3. Greedy takes the largest coin that fits: 1, 0 left
4. Greedy used 3 coins, but 3 + 3 needs only 2 → greedy is wrong here

</details>


## Lesson 2: Proving a greedy: the exchange argument

*Level: Core*

To show a greedy choice is safe, take **any** optimal answer and show you can swap one of its choices for the greedy choice without making it worse. Then some optimal answer agrees with greedy, step by step.



Example — **assign cookies**: give the smallest cookie that satisfies the least greedy child. If an optimal answer gives that child a bigger cookie, swapping it for the smaller one never hurts anyone else.



Greedy problems usually start with the right **sort order**:




- Two city scheduling: by `costA − costB`.
- Maximum units on a truck: by units per box, high to low.
- Largest number: by the rule "a before b if a + b &gt; b + a".
- Queue by height: tallest first, then insert at index k.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q17](#q17), [Q3](#q3), [Q14](#q14), [Q15](#q15). Try one now, or carry on and do them in Part 3.

## Lesson 3: Intervals: sort, then merge

*Level: Core*

An interval is `[start, end]`. Two intervals overlap when `a.start <= b.end` and `b.start <= a.end`. Sorted by start, you only need to compare each interval with the last merged one:



```python
intervals.sort()
out = [intervals[0]]
for s, e in intervals[1:]:
    if s <= out[-1][1]:            # overlaps
        out[-1][1] = max(out[-1][1], e)
    else:
        out.append([s, e])
```



Insert interval is the same in one pass on an already sorted list: copy the ones before, absorb the overlapping ones, copy the rest. Intersections of two sorted lists use two pointers.

**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q8](#q8), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 4: Scheduling: sort by end time

*Level: Core*

"Keep as many non-overlapping intervals as possible" (or "remove as few as possible"): sort by **end** and always keep the interval that finishes first. Finishing early leaves the most room for the rest — that's the exchange argument.



```python
intervals.sort(key=lambda x: x[1])
end, kept = -inf, 0
for s, e in intervals:
    if s >= end:
        kept += 1; end = e
```



Same idea: **minimum arrows** to burst balloons (shoot at the first end), and **covered intervals** (sort by start, longer first, track the farthest end).



**⚠️ Watch out:** Sorting by start gives the wrong answer here — a long early interval would block many short ones.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def scheduling_lesson():
    meetings = [[1, 4], [2, 3], [3, 5], [4, 7], [6, 8], [5, 9]]
    meetings.sort(key=lambda x: x[1])
    end, kept = float("-inf"), []
    for i, (s, e) in enumerate(meetings):
        if s >= end:
            kept.append(i)
            end = e
        else:
            pass
    return len(kept)


scheduling_lesson()

3

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. [2,3] starts after nothing → keep it
2. [1,4] overlaps the last kept meeting → skip
3. [3,5] starts after the last end → keep it
4. [4,7] overlaps the last kept meeting → skip
5. [6,8] starts after the last end → keep it
6. [5,9] overlaps the last kept meeting → skip

</details>


**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q10](#q10), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 5: Reach and running balance

*Level: Core*

- **Jump game**: track the furthest index reachable so far. If you stand past it, you're stuck.
- **Jump game II**: like BFS — the current reach window is "reachable in k jumps"; the furthest point from it starts the next window.
- **Gas station**: if the tank goes negative at station i, no start between the current start and i can work — restart at i + 1.
- **Valid parenthesis string**: keep the range [lo, hi] of possible open counts.
- **Stock II**: add every positive day-to-day rise.
- **Lemonade change**: give the biggest bills first to keep small ones.

**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q5](#q5), [Q6](#q6), [Q12](#q12), [Q13](#q13), [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 6: Sweep lines and two passes

*Level: Advanced*

#### Sweep line



Turn intervals into **events**: `+1` at a start, `−1` at an end. Sort events by time (ends before starts at the same time when touching doesn't overlap) and walk them with a running total. The peak is the max overlap — rooms needed, passengers in the car.



#### Two passes



When a rule involves both neighbours, satisfy one side at a time. Candy: left → right gives more than the left neighbour; right → left fixes the right neighbour while keeping the max.



#### Last occurrence



Partition labels: record each letter's last index; a part can end only when you reach the furthest last index of the letters inside it.

**✏️ Your turn:** you now have what you need for [Q19](#q19), [Q20](#q20), [Q11](#q11). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Assign Cookies

🟢 Warm-up · *Sort both, smallest that fits*

Child i is content with a cookie of size ≥ `g[i]`. Each child gets at most one cookie. Maximise the content children.

**Examples**

- `g = [1, 2, 3], s = [1, 1]` → `1`
- `g = [1, 2], s = [1, 2, 3]` → `2`

In [ ]:
def find_content_children(g, s):
    """
    Child i is content with a cookie of size ≥ g[i]. Each child gets at most one
    cookie. Maximise the content children.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_content_children, [
    (([1, 2, 3], [1, 1]), 1),
    (([1, 2], [1, 2, 3]), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each child, scan every unused cookie for the smallest that fits.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort children and cookies. Walk the cookies; give the current cookie to the least greedy unfed child if it fits.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each child, scan every unused cookie for the smallest that fits.

```python
def find_content_children(g, s):
    used, count = [False] * len(s), 0
    for need in sorted(g):
        best = -1
        for j, size in enumerate(s):
            if not used[j] and size >= need and (best < 0 or size < s[best]):
                best = j
        if best >= 0:
            used[best] = True
            count += 1
    return count
```

**Cost:** O(n · m) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort children and cookies. Walk the cookies; give the current cookie to the least greedy unfed child if it fits.

```python
def find_content_children(g, s):
    g.sort()
    s.sort()
    child = 0
    for j, size in enumerate(s):
        if child < len(g) and size >= g[child]:
            child += 1
    return child
```

**Cost:** O(n log n + m log m) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each child, scan every unused cookie for the smallest that fits. That costs O(n · m) time and O(m) space."

"I can do better: Sort children and cookies. Walk the cookies; give the current cookie to the least greedy unfed child if it fits. That's O(n log n + m log m) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `g = [1, 2, 3, 4], s = [1, 2, 2, 5]`

1. Cookie 1: children fed so far 1
2. Cookie 2: children fed so far 2
3. Cookie 2: children fed so far 2
4. Cookie 5: children fed so far 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/assign-cookies)

<a id="q2"></a>
### Q2. Lemonade Change

🟢 Warm-up · *Give the biggest change first*

Lemonade costs 5. Customers pay with 5, 10 or 20 in order. Starting with no money, can you give everyone correct change?

**Examples**

- `bills = [5, 5, 5, 10, 20]` → `True`
- `bills = [5, 5, 10, 10, 20]` → `False`

In [ ]:
def lemonade_change(bills):
    """
    Lemonade costs 5. Customers pay with 5, 10 or 20 in order. Starting with no
    money, can you give everyone correct change?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lemonade_change, [
    (([5, 5, 5, 10, 20],), True),
    (([5, 5, 10, 10, 20],), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every way to make change from the bills you hold (search).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count 5s and 10s. For 20, prefer 10 + 5 (keeps more 5s, which are needed for every change) and fall back to 5 + 5 + 5.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every way to make change from the bills you hold (search).

```python
def lemonade_change(bills):
    def go(i, fives, tens):
        if i == len(bills):
            return True
        b = bills[i]
        if b == 5:
            return go(i + 1, fives + 1, tens)
        if b == 10:
            return fives > 0 and go(i + 1, fives - 1, tens + 1)
        return (tens > 0 and fives > 0 and go(i + 1, fives - 1, tens - 1)) or (fives >= 3 and go(i + 1, fives - 3, tens))
    return go(0, 0, 0)
```

**Cost:** O(2ⁿ) worst time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Count 5s and 10s. For 20, prefer 10 + 5 (keeps more 5s, which are needed for every change) and fall back to 5 + 5 + 5.

```python
def lemonade_change(bills):
    fives = tens = 0
    for i, b in enumerate(bills):
        if b == 5:
            fives += 1
        elif b == 10:
            fives, tens = fives - 1, tens + 1
        elif tens:
            fives, tens = fives - 1, tens - 1
        else:
            fives -= 3
        if fives < 0:
            return False
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every way to make change from the bills you hold (search). That costs O(2ⁿ) worst time and O(n) space."

"I can do better: Count 5s and 10s. For 20, prefer 10 + 5 (keeps more 5s, which are needed for every change) and fall back to 5 + 5 + 5. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `bills = [5, 5, 5, 10, 20]`

1. Customer pays 5
2. Customer pays 5
3. Customer pays 5
4. Customer pays 10
5. Customer pays 20

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/lemonade-change)

<a id="q3"></a>
### Q3. Maximum Units on a Truck

🟢 Warm-up · *Best units per box first*

`boxTypes[i] = [count, unitsPerBox]`. The truck holds `truckSize` boxes. Maximise total units.

**Examples**

- `box_types = [[1, 3], [2, 2], [3, 1]], truck_size = 4` → `8`
- `box_types = [[5, 10], [2, 5], [4, 7], [3, 9]], truck_size = 10` → `91`

In [ ]:
def maximum_units(box_types, truck_size):
    """
    boxTypes[i] = [count, unitsPerBox]. The truck holds truckSize boxes.
    Maximise total units.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(maximum_units, [
    (([[1, 3], [2, 2], [3, 1]], 4), 8),
    (([[5, 10], [2, 5], [4, 7], [3, 9]], 10), 91),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Load boxes one at a time, each time scanning for the type with the most units left.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort box types by units per box, descending, and take as many of each as fit.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Load boxes one at a time, each time scanning for the type with the most units left.

```python
def maximum_units(box_types, truck_size):
    left = [list(b) for b in box_types]
    total = 0
    for _ in range(truck_size):
        best = max((b for b in left if b[0] > 0), key=lambda b: b[1], default=None)
        if not best:
            break
        best[0] -= 1
        total += best[1]
    return total
```

**Cost:** O(truckSize · n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort box types by units per box, descending, and take as many of each as fit.

```python
def maximum_units(box_types, truck_size):
    box_types.sort(key=lambda b: -b[1])
    total = 0
    for i, (count, units) in enumerate(box_types):
        take = min(count, truck_size)
        total += take * units
        truck_size -= take
        if truck_size == 0:
            break
    return total
```

**Cost:** O(n log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Load boxes one at a time, each time scanning for the type with the most units left. That costs O(truckSize · n) time and O(1) space."

"I can do better: Sort box types by units per box, descending, and take as many of each as fit. That's O(n log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `box_types = [[1, 3], [2, 2], [3, 1]], truck_size = 4`

1. Take 1 box(es) of 3 units
2. Take 2 box(es) of 2 units
3. Take 1 box(es) of 1 units

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/maximum-units-on-a-truck)

<a id="q4"></a>
### Q4. Jump Game

🟡 Interview standard · *Furthest reach*

`nums[i]` is the maximum jump length from index i. Can you reach the last index from index 0?

**Examples**

- `nums = [2, 3, 1, 1, 4]` → `True`
- `nums = [3, 2, 1, 0, 4]` → `False`

In [ ]:
def can_jump(nums):
    """
    nums[i] is the maximum jump length from index i. Can you reach the last
    index from index 0?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(can_jump, [
    (([2, 3, 1, 1, 4],), True),
    (([3, 2, 1, 0, 4],), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursively try every jump length from every index.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Track the furthest index reachable. If you stand beyond it, you're stuck; otherwise extend it with i + nums[i].

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recursively try every jump length from every index.

```python
def can_jump(nums):
    def go(i):
        if i >= len(nums) - 1:
            return True
        return any(go(i + j) for j in range(1, nums[i] + 1))
    return go(0)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Track the furthest index reachable. If you stand beyond it, you're stuck; otherwise extend it with i + nums[i].

```python
def can_jump(nums):
    reach = 0
    for i, x in enumerate(nums):
        if i > reach:
            return False
        reach = max(reach, i + x)
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursively try every jump length from every index. That costs O(2ⁿ) time and O(n) space."

"I can do better: Track the furthest index reachable. If you stand beyond it, you're stuck; otherwise extend it with i + nums[i]. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [2, 3, 1, 1, 4]`

1. From 0 we can reach up to 2
2. From 1 we can reach up to 4
3. From 2 we can reach up to 4
4. From 3 we can reach up to 4
5. From 4 we can reach up to 8

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/jump-game)

<a id="q5"></a>
### Q5. Jump Game II

🟡 Interview standard · *BFS by reach windows*

Return the minimum number of jumps to reach the last index (it's always reachable).

**Examples**

- `nums = [2, 3, 1, 1, 4]` → `2`
- `nums = [2, 3, 0, 1, 4]` → `2`

In [ ]:
def jump(nums):
    """
    Return the minimum number of jumps to reach the last index (it's always
    reachable).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(jump, [
    (([2, 3, 1, 1, 4],), 2),
    (([2, 3, 0, 1, 4],), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

dp[i] = 1 + min(dp[j]) over every j that can jump to i.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Treat each jump as a BFS layer: the current window [start, end] is reachable in k jumps; the furthest index reachable from it is the end of the next window.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** dp[i] = 1 + min(dp[j]) over every j that can jump to i.

```python
def jump(nums):
    n = len(nums)
    dp = [0] + [float("inf")] * (n - 1)
    for i in range(1, n):
        for j in range(i):
            if j + nums[j] >= i:
                dp[i] = min(dp[i], dp[j] + 1)
    return dp[-1]
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Treat each jump as a BFS layer: the current window [start, end] is reachable in k jumps; the furthest index reachable from it is the end of the next window.

```python
def jump(nums):
    jumps, end, far = 0, 0, 0
    for i in range(len(nums) - 1):
        far = max(far, i + nums[i])
        if i == end:
            jumps += 1
            end = far
    return jumps
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: dp[i] = 1 + min(dp[j]) over every j that can jump to i. That costs O(n²) time and O(n) space."

"I can do better: Treat each jump as a BFS layer: the current window [start, end] is reachable in k jumps; the furthest index reachable from it is the end of the next window. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [2, 3, 1, 1, 2, 1, 4]`

1. Window ends at 0: jump #1 lets us reach up to 2
2. Window ends at 2: jump #2 lets us reach up to 4
3. Window ends at 4: jump #3 lets us reach up to 6

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/jump-game-ii)

<a id="q6"></a>
### Q6. Gas Station

🟡 Interview standard · *Running tank, restart on deficit*

Station i gives `gas[i]`; driving to the next costs `cost[i]`. Return the start index that completes the circle, or −1.

**Examples**

- `gas = [1, 2, 3, 4, 5], cost = [3, 4, 5, 1, 2]` → `3`
- `gas = [2, 3, 4], cost = [3, 4, 3]` → `-1`

In [ ]:
def can_complete_circuit(gas, cost):
    """
    Station i gives gas[i]; driving to the next costs cost[i]. Return the start
    index that completes the circle, or −1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(can_complete_circuit, [
    (([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]), 3),
    (([2, 3, 4], [3, 4, 3]), -1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate the full loop from every start.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

If total gas &lt; total cost it's impossible. Otherwise drive once; when the tank goes negative at i, no station from the current start to i can work, so restart at i + 1.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Simulate the full loop from every start.

```python
def can_complete_circuit(gas, cost):
    n = len(gas)
    for s in range(n):
        tank = 0
        for k in range(n):
            i = (s + k) % n
            tank += gas[i] - cost[i]
            if tank < 0:
                break
        else:
            return s
    return -1
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** If total gas &lt; total cost it's impossible. Otherwise drive once; when the tank goes negative at i, no station from the current start to i can work, so restart at i + 1.

```python
def can_complete_circuit(gas, cost):
    if sum(gas) < sum(cost):
        return -1
    start = tank = 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:
            start, tank = i + 1, 0
        else:
            pass
    return start
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate the full loop from every start. That costs O(n²) time and O(1) space."

"I can do better: If total gas &lt; total cost it's impossible. Otherwise drive once; when the tank goes negative at i, no station from the current start to i can work, so restart at i + 1. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `gas = [1, 2, 3, 4, 5], cost = [3, 4, 5, 1, 2]`

1. Tank went negative at 0 → no start up to 0 works; restart at 1
2. Tank went negative at 1 → no start up to 1 works; restart at 2
3. Tank went negative at 2 → no start up to 2 works; restart at 3
4. Station 3: tank 3
5. Station 4: tank 6

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/gas-station)

<a id="q7"></a>
### Q7. Merge Intervals

🟡 Interview standard · *Sort by start*

Merge all overlapping intervals.

**Examples**

- `intervals = [[1, 3], [2, 6], [8, 10], [15, 18]]` → `[[1, 6], [8, 10], [15, 18]]`
- `intervals = [[1, 4], [4, 5]]` → `[[1, 5]]`

In [ ]:
def merge_intervals(intervals):
    """
    Merge all overlapping intervals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(merge_intervals, [
    (([[1, 3], [2, 6], [8, 10], [15, 18]],), [[1, 6], [8, 10], [15, 18]]),
    (([[1, 4], [4, 5]],), [[1, 5]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly look for any two overlapping intervals and merge them until none overlap.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by start; extend the last merged interval while the next starts before it ends, otherwise start a new one.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Repeatedly look for any two overlapping intervals and merge them until none overlap.

```python
def merge_intervals(intervals):
    res = [list(x) for x in intervals]
    merged = True
    while merged:
        merged = False
        for i in range(len(res)):
            for j in range(i + 1, len(res)):
                a, b = res[i], res[j]
                if a[0] <= b[1] and b[0] <= a[1]:
                    res[i] = [min(a[0], b[0]), max(a[1], b[1])]
                    res.pop(j)
                    merged = True
                    break
            if merged:
                break
    return sorted(res)
```

**Cost:** O(n³) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort by start; extend the last merged interval while the next starts before it ends, otherwise start a new one.

```python
def merge_intervals(intervals):
    intervals.sort()
    out = [intervals[0][:]]
    for s, e in intervals[1:]:
        if s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly look for any two overlapping intervals and merge them until none overlap. That costs O(n³) time and O(n) space."

"I can do better: Sort by start; extend the last merged interval while the next starts before it ends, otherwise start a new one. That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `intervals = [[1, 3], [2, 6], [8, 10], [15, 18]]`

1. Process [2,6]
2. Process [8,10]
3. Process [15,18]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/merge-intervals)

<a id="q8"></a>
### Q8. Insert Interval

🟡 Interview standard · *Three phases: before, overlap, after*

Insert `newInterval` into a sorted list of non-overlapping intervals, merging where needed.

**Examples**

- `intervals = [[1, 3], [6, 9]], new_interval = [2, 5]` → `[[1, 5], [6, 9]]`
- `intervals = [[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], new_interval = [4, 8]` → `[[1, 2], [3, 10], [12, 16]]`

In [ ]:
def insert(intervals, new_interval):
    """
    Insert newInterval into a sorted list of non-overlapping intervals, merging
    where needed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(insert, [
    (([[1, 3], [6, 9]], [2, 5]), [[1, 5], [6, 9]]),
    (([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]), [[1, 2], [3, 10], [12, 16]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Append the new interval, sort everything, and run Merge Intervals.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One pass: copy intervals ending before it, absorb every interval overlapping it into one, copy the rest.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Append the new interval, sort everything, and run Merge Intervals.

```python
def insert(intervals, new_interval):
    allv = sorted(intervals + [new_interval])
    out = [allv[0]]
    for s, e in allv[1:]:
        if s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** One pass: copy intervals ending before it, absorb every interval overlapping it into one, copy the rest.

```python
def insert(intervals, new_interval):
    s, e = new_interval
    out, i, n = [], 0, len(intervals)
    while i < n and intervals[i][1] < s:
        out.append(intervals[i])
        i += 1
    while i < n and intervals[i][0] <= e:
        s, e = min(s, intervals[i][0]), max(e, intervals[i][1])
        i += 1
    out.append([s, e])
    out += intervals[i:]
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Append the new interval, sort everything, and run Merge Intervals. That costs O(n log n) time and O(n) space."

"I can do better: One pass: copy intervals ending before it, absorb every interval overlapping it into one, copy the rest. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `intervals = [[1, 3], [6, 9]], new_interval = [2, 5]`

1. 1) Copy intervals that end before the new one starts
2. 2) Absorb [1,3] → [1,5]
3. 3) Add the merged interval and copy the rest

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/insert-interval)

<a id="q9"></a>
### Q9. Non-overlapping Intervals

🟡 Interview standard · *Sort by end, keep earliest finish*

Return the minimum number of intervals to remove so the rest don't overlap (touching ends are fine).

**Examples**

- `intervals = [[1, 2], [2, 3], [3, 4], [1, 3]]` → `1`
- `intervals = [[1, 2], [1, 2], [1, 2]]` → `2`
- `intervals = [[1, 2], [2, 3]]` → `0`

In [ ]:
def erase_overlap_intervals(intervals):
    """
    Return the minimum number of intervals to remove so the rest don't overlap
    (touching ends are fine).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(erase_overlap_intervals, [
    (([[1, 2], [2, 3], [3, 4], [1, 3]],), 1),
    (([[1, 2], [1, 2], [1, 2]],), 2),
    (([[1, 2], [2, 3]],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every subset of intervals to keep and check it's non-overlapping.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by end time. Keep an interval if it starts at or after the last kept end; otherwise remove it. Finishing earliest leaves the most room.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every subset of intervals to keep and check it's non-overlapping.

```python
def erase_overlap_intervals(intervals):
    n, best = len(intervals), 0
    for mask in range(1 << n):
        keep = sorted(intervals[i] for i in range(n) if mask >> i & 1)
        if all(keep[k][1] <= keep[k + 1][0] for k in range(len(keep) - 1)):
            best = max(best, len(keep))
    return n - best
```

**Cost:** O(n · 2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort by end time. Keep an interval if it starts at or after the last kept end; otherwise remove it. Finishing earliest leaves the most room.

```python
def erase_overlap_intervals(intervals):
    intervals.sort(key=lambda x: x[1])
    end, removed, kept = float("-inf"), 0, []
    for i, (s, e) in enumerate(intervals):
        if s >= end:
            end = e
            kept.append(i)
        else:
            removed += 1
    return removed
```

**Cost:** O(n log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every subset of intervals to keep and check it's non-overlapping. That costs O(n · 2ⁿ) time and O(n) space."

"I can do better: Sort by end time. Keep an interval if it starts at or after the last kept end; otherwise remove it. Finishing earliest leaves the most room. That's O(n log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `intervals = [[1, 4], [2, 3], [3, 5], [4, 6], [5, 9]]`

1. [2,3] keep
2. [1,4] overlaps → remove
3. [3,5] keep
4. [4,6] overlaps → remove
5. [5,9] keep

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/non-overlapping-intervals)

<a id="q10"></a>
### Q10. Minimum Number of Arrows to Burst Balloons

🟡 Interview standard · *Sort by end, shoot at the end*

Each balloon spans [x\_start, x\_end]. An arrow at x bursts every balloon with start ≤ x ≤ end. Return the fewest arrows.

**Examples**

- `points = [[10, 16], [2, 8], [1, 6], [7, 12]]` → `2`
- `points = [[1, 2], [3, 4], [5, 6], [7, 8]]` → `4`
- `points = [[1, 2], [2, 3], [3, 4], [4, 5]]` → `2`

In [ ]:
def find_min_arrow_shots(points):
    """
    Each balloon spans [x_start, x_end]. An arrow at x bursts every balloon with
    start ≤ x ≤ end. Return the fewest arrows.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_min_arrow_shots, [
    (([[10, 16], [2, 8], [1, 6], [7, 12]],), 2),
    (([[1, 2], [3, 4], [5, 6], [7, 8]],), 4),
    (([[1, 2], [2, 3], [3, 4], [4, 5]],), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every subset of candidate arrow positions (the balloon ends) and keep the smallest that hits all.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by end. Shoot at the first balloon's end; every balloon starting at or before that point is burst. The next unburst balloon needs a new arrow at its end.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every subset of candidate arrow positions (the balloon ends) and keep the smallest that hits all.

```python
def find_min_arrow_shots(points):
    xs = sorted({e for _, e in points})
    for k in range(1, len(xs) + 1):
        from itertools import combinations
        for shots in combinations(xs, k):
            if all(any(s <= x <= e for x in shots) for s, e in points):
                return k
    return 0
```

**Cost:** O(n · 2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort by end. Shoot at the first balloon's end; every balloon starting at or before that point is burst. The next unburst balloon needs a new arrow at its end.

```python
def find_min_arrow_shots(points):
    points.sort(key=lambda p: p[1])
    arrows, x = 0, float("-inf")
    for i, (s, e) in enumerate(points):
        if s > x:
            arrows += 1
            x = e
        else:
            pass
    return arrows
```

**Cost:** O(n log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every subset of candidate arrow positions (the balloon ends) and keep the smallest that hits all. That costs O(n · 2ⁿ) time and O(n) space."

"I can do better: Sort by end. Shoot at the first balloon's end; every balloon starting at or before that point is burst. The next unburst balloon needs a new arrow at its end. That's O(n log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `points = [[10, 16], [2, 8], [1, 6], [7, 12]]`

1. Balloon [1,6] not hit → new arrow at 6
2. Balloon [2,8] already burst by the arrow at 6
3. Balloon [7,12] not hit → new arrow at 12
4. Balloon [10,16] already burst by the arrow at 12

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/minimum-number-of-arrows-to-burst-balloons)

<a id="q11"></a>
### Q11. Partition Labels

🟡 Interview standard · *Last occurrence*

Split `s` into as many parts as possible so each letter appears in at most one part. Return the part sizes.

**Examples**

- `s = 'ababcbacadefegdehijhklij'` → `[9, 7, 8]`
- `s = 'eccbbbbdec'` → `[10]`

In [ ]:
def partition_labels(s):
    """
    Split s into as many parts as possible so each letter appears in at most one
    part. Return the part sizes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(partition_labels, [
    (('ababcbacadefegdehijhklij',), [9, 7, 8]),
    (('eccbbbbdec',), [10]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each cut position, check that no letter appears on both sides; cut wherever it's safe.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Record each letter's last index. Walk s extending the current part's end to the last index of every letter seen; when i reaches the end, cut.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each cut position, check that no letter appears on both sides; cut wherever it's safe.

```python
def partition_labels(s):
    sizes, start = [], 0
    for i in range(len(s)):
        if not set(s[start:i + 1]) & set(s[i + 1:]):
            sizes.append(i - start + 1)
            start = i + 1
    return sizes
```

**Cost:** O(n²) time, O(σ) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Record each letter's last index. Walk s extending the current part's end to the last index of every letter seen; when i reaches the end, cut.

```python
def partition_labels(s):
    last = {ch: i for i, ch in enumerate(s)}
    sizes, start, end = [], 0, 0
    for i, ch in enumerate(s):
        end = max(end, last[ch])
        if i == end:
            sizes.append(end - start + 1)
            start = i + 1
    return sizes
```

**Cost:** O(n) time, O(σ) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each cut position, check that no letter appears on both sides; cut wherever it's safe. That costs O(n²) time and O(σ) space."

"I can do better: Record each letter's last index. Walk s extending the current part's end to the last index of every letter seen; when i reaches the end, cut. That's O(n) time and O(σ) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'abacdcbef'`

1. 'a' last appears at 2 → this part ends at least at 2
2. 'b' last appears at 6 → this part ends at least at 6
3. 'a' last appears at 2 → this part ends at least at 6
4. 'c' last appears at 5 → this part ends at least at 6
5. 'd' last appears at 4 → this part ends at least at 6
6. 'c' last appears at 5 → this part ends at least at 6
7. 'b' last appears at 6 → this part ends at least at 6
8. 'e' last appears at 7 → this part ends at least at 7
9. 'f' last appears at 8 → this part ends at least at 8

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/partition-labels)

<a id="q12"></a>
### Q12. Valid Parenthesis String

🟡 Interview standard · *Range of open counts*

'\*' can be '(' or ')' or empty. Is the string a valid parenthesis string?

**Examples**

- `s = '()'` → `True`
- `s = '(*)'` → `True`
- `s = '(*))'` → `True`

In [ ]:
def check_valid_string(s):
    """
    '*' can be '(' or ')' or empty. Is the string a valid parenthesis string?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(check_valid_string, [
    (('()',), True),
    (('(*)',), True),
    (('(*))',), True),
    ((')*(',), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try all three meanings of every '\*'.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Track the range [lo, hi] of possible open-bracket counts. '(' raises both, ')' lowers both, '\*' lowers lo and raises hi. If hi &lt; 0 fail; clamp lo at 0; valid if lo == 0 at the end.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try all three meanings of every '\*'.

```python
def check_valid_string(s):
    def go(i, open_):
        if open_ < 0:
            return False
        if i == len(s):
            return open_ == 0
        if s[i] == "(":
            return go(i + 1, open_ + 1)
        if s[i] == ")":
            return go(i + 1, open_ - 1)
        return go(i + 1, open_ + 1) or go(i + 1, open_ - 1) or go(i + 1, open_)
    return go(0, 0)
```

**Cost:** O(3ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Track the range [lo, hi] of possible open-bracket counts. '(' raises both, ')' lowers both, '\*' lowers lo and raises hi. If hi &lt; 0 fail; clamp lo at 0; valid if lo == 0 at the end.

```python
def check_valid_string(s):
    lo = hi = 0
    for i, ch in enumerate(s):
        if ch == "(":
            lo, hi = lo + 1, hi + 1
        elif ch == ")":
            lo, hi = lo - 1, hi - 1
        else:
            lo, hi = lo - 1, hi + 1
        if hi < 0:
            return False
        lo = max(lo, 0)
    return lo == 0
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try all three meanings of every '\*'. That costs O(3ⁿ) time and O(n) space."

"I can do better: Track the range [lo, hi] of possible open-bracket counts. '(' raises both, ')' lowers both, '\*' lowers lo and raises hi. If hi &lt; 0 fail; clamp lo at 0; valid if lo == 0 at the end. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = '(*))(*'`

1. '(': open count could be anywhere in [1, 1]
2. '\*': open count could be anywhere in [0, 2]
3. ')': open count could be anywhere in [0, 1]
4. ')': open count could be anywhere in [0, 0]
5. '(': open count could be anywhere in [1, 1]
6. '\*': open count could be anywhere in [0, 2]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/valid-parenthesis-string)

<a id="q13"></a>
### Q13. Best Time to Buy and Sell Stock II

🟡 Interview standard · *Take every rise*

You may trade any number of times (hold at most one share). Maximise profit.

**Examples**

- `prices = [7, 1, 5, 3, 6, 4]` → `7`
- `prices = [1, 2, 3, 4, 5]` → `4`
- `prices = [7, 6, 4, 3, 1]` → `0`

In [ ]:
def max_profit_ii(prices):
    """
    You may trade any number of times (hold at most one share). Maximise profit.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_profit_ii, [
    (([7, 1, 5, 3, 6, 4],), 7),
    (([1, 2, 3, 4, 5],), 4),
    (([7, 6, 4, 3, 1],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recurse on (day, holding) trying buy, sell or wait.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Any profitable trade is a sum of day-to-day rises, so add every positive difference prices[i] − prices[i − 1].

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recurse on (day, holding) trying buy, sell or wait.

```python
def max_profit_ii(prices):
    def go(i, holding):
        if i == len(prices):
            return 0
        wait = go(i + 1, holding)
        if holding:
            return max(wait, prices[i] + go(i + 1, False))
        return max(wait, -prices[i] + go(i + 1, True))
    return go(0, False)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Any profitable trade is a sum of day-to-day rises, so add every positive difference prices[i] − prices[i − 1].

```python
def max_profit_ii(prices):
    profit = 0
    for i in range(1, len(prices)):
        gain = prices[i] - prices[i - 1]
        if gain > 0:
            profit += gain
    return profit
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recurse on (day, holding) trying buy, sell or wait. That costs O(2ⁿ) time and O(n) space."

"I can do better: Any profitable trade is a sum of day-to-day rises, so add every positive difference prices[i] − prices[i − 1]. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `prices = [7, 1, 5, 3, 6, 4]`

1. Day 0 → 1: change -6
2. Day 1 → 2: change +4 (take it)
3. Day 2 → 3: change -2
4. Day 3 → 4: change +3 (take it)
5. Day 4 → 5: change -2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/best-time-to-buy-and-sell-stock-ii)

<a id="q14"></a>
### Q14. Largest Number

🟡 Interview standard · *Custom comparator a+b vs b+a*

Arrange non-negative integers to form the largest number; return it as a string.

**Examples**

- `nums = [10, 2]` → `'210'`
- `nums = [3, 30, 34, 5, 9]` → `'9534330'`
- `nums = [0, 0]` → `'0'`

In [ ]:
def largest_number(nums):
    """
    Arrange non-negative integers to form the largest number; return it as a
    string.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(largest_number, [
    (([10, 2],), '210'),
    (([3, 30, 34, 5, 9],), '9534330'),
    (([0, 0],), '0'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every permutation and keep the largest concatenation.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort with the rule: a goes before b if a + b &gt; b + a as strings. This order is transitive, and it gives the largest number. Handle all zeros.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every permutation and keep the largest concatenation.

```python
def largest_number(nums):
    from itertools import permutations
    best = max("".join(p) for p in permutations(map(str, nums)))
    return "0" if best[0] == "0" else best
```

**Cost:** O(n! · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort with the rule: a goes before b if a + b &gt; b + a as strings. This order is transitive, and it gives the largest number. Handle all zeros.

```python
def largest_number(nums):
    from functools import cmp_to_key
    strs = [str(x) for x in nums]
    strs.sort(key=cmp_to_key(lambda a, b: -1 if a + b > b + a else (1 if a + b < b + a else 0)))
    res = "".join(strs)
    return "0" if res[0] == "0" else res
```

**Cost:** O(n log n · k) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every permutation and keep the largest concatenation. That costs O(n! · n) time and O(n) space."

"I can do better: Sort with the rule: a goes before b if a + b &gt; b + a as strings. This order is transitive, and it gives the largest number. Handle all zeros. That's O(n log n · k) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/largest-number)

<a id="q15"></a>
### Q15. Queue Reconstruction by Height

🟡 Interview standard · *Tallest first, insert at k*

Each person is [h, k]: height h with exactly k people in front who are at least as tall. Rebuild the queue.

**Examples**

- `people = [[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]]` → `[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]`
- `people = [[6, 0], [5, 0], [4, 0], [3, 2], [2, 2], [1, 4]]` → `[[4, 0], [5, 0], [2, 2], [3, 2], [1, 4], [6, 0]]`

In [ ]:
def reconstruct_queue(people):
    """
    Each person is [h, k]: height h with exactly k people in front who are at
    least as tall. Rebuild the queue.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reconstruct_queue, [
    (([[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]],), [[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]),
    (([[6, 0], [5, 0], [4, 0], [3, 2], [2, 2], [1, 4]],), [[4, 0], [5, 0], [2, 2], [3, 2], [1, 4], [6, 0]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every ordering and check each person's k.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by height descending, k ascending. Insert each person at index k: everyone already placed is at least as tall, and shorter people added later don't change their counts.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every ordering and check each person's k.

```python
def reconstruct_queue(people):
    from itertools import permutations
    for p in permutations(people):
        if all(sum(1 for j in range(i) if p[j][0] >= h) == k for i, (h, k) in enumerate(p)):
            return [list(x) for x in p]
```

**Cost:** O(n! · n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort by height descending, k ascending. Insert each person at index k: everyone already placed is at least as tall, and shorter people added later don't change their counts.

```python
def reconstruct_queue(people):
    people.sort(key=lambda p: (-p[0], p[1]))
    queue = []
    for h, k in people:
        queue.insert(k, [h, k])
    return queue
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every ordering and check each person's k. That costs O(n! · n²) time and O(n) space."

"I can do better: Sort by height descending, k ascending. Insert each person at index k: everyone already placed is at least as tall, and shorter people added later don't change their counts. That's O(n²) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `people = [[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]]`

1. Insert [7,0] at index 0
2. Insert [7,1] at index 1
3. Insert [6,1] at index 1
4. Insert [5,0] at index 0
5. Insert [5,2] at index 2
6. Insert [4,4] at index 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/queue-reconstruction-by-height)

<a id="q16"></a>
### Q16. Interval List Intersections

🟡 Interview standard · *Two pointers on intervals*

Two lists of disjoint, sorted closed intervals. Return their intersection.

**Examples**

- `first = [[0, 2], [5, 10], [13, 23], [24, 25]], second = [[1, 5], [8, 12], [15, 24], [25, 26]]` → `[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]`
- `first = [[1, 3], [5, 9]], second = []` → `[]`

In [ ]:
def interval_intersection(first, second):
    """
    Two lists of disjoint, sorted closed intervals. Return their intersection.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(interval_intersection, [
    (([[0, 2], [5, 10], [13, 23], [24, 25]], [[1, 5], [8, 12], [15, 24], [25, 26]]), [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]),
    (([[1, 3], [5, 9]], []), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Intersect every interval in A with every interval in B.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two pointers: the overlap of A[i] and B[j] is [max starts, min ends] if non-empty. Then advance whichever interval ends first.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Intersect every interval in A with every interval in B.

```python
def interval_intersection(first, second):
    res = []
    for a, b in first:
        for c, d in second:
            lo, hi = max(a, c), min(b, d)
            if lo <= hi:
                res.append([lo, hi])
    return sorted(res)
```

**Cost:** O(m · n) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Two pointers: the overlap of A[i] and B[j] is [max starts, min ends] if non-empty. Then advance whichever interval ends first.

```python
def interval_intersection(first, second):
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        lo = max(first[i][0], second[j][0])
        hi = min(first[i][1], second[j][1])
        if lo <= hi:
            res.append([lo, hi])
        if first[i][1] < second[j][1]:
            i += 1
        else:
            j += 1
    return res
```

**Cost:** O(m + n) time, O(1) extra space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Intersect every interval in A with every interval in B. That costs O(m · n) time and O(m + n) space."

"I can do better: Two pointers: the overlap of A[i] and B[j] is [max starts, min ends] if non-empty. Then advance whichever interval ends first. That's O(m + n) time and O(1) extra space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `first = [[0, 2], [5, 10], [13, 23], [24, 25]], second = [[1, 5], [8, 12], [15, 24], [25, 26]]`

1. Overlap of [0, 2] and [1, 5]: [1,2]
2. Overlap of [5, 10] and [1, 5]: [5,5]
3. Overlap of [5, 10] and [8, 12]: [8,10]
4. Overlap of [13, 23] and [8, 12]: none
5. Overlap of [13, 23] and [15, 24]: [15,23]
6. Overlap of [24, 25] and [15, 24]: [24,24]
7. Overlap of [24, 25] and [25, 26]: [25,25]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/interval-list-intersections)

<a id="q17"></a>
### Q17. Two City Scheduling

🟡 Interview standard · *Sort by cost difference*

2n people; `costs[i] = [toA, toB]`. Send exactly n to each city at minimum total cost.

**Examples**

- `costs = [[10, 20], [30, 200], [400, 50], [30, 20]]` → `110`
- `costs = [[259, 770], [448, 54], [926, 667], [184, 139], [840, 118], [577, 469]]` → `1859`

In [ ]:
def two_city_sched_cost(costs):
    """
    2n people; costs[i] = [toA, toB]. Send exactly n to each city at minimum
    total cost.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(two_city_sched_cost, [
    (([[10, 20], [30, 200], [400, 50], [30, 20]],), 110),
    (([[259, 770], [448, 54], [926, 667], [184, 139], [840, 118], [577, 469]],), 1859),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every way to choose n people for city A.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by toA − toB. The first n (who save the most by going to A) go to A, the rest to B.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every way to choose n people for city A.

```python
def two_city_sched_cost(costs):
    from itertools import combinations
    n = len(costs) // 2
    best = float("inf")
    for group in combinations(range(2 * n), n):
        g = set(group)
        best = min(best, sum(costs[i][0] if i in g else costs[i][1] for i in range(2 * n)))
    return best
```

**Cost:** O(C(2n, n) · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort by toA − toB. The first n (who save the most by going to A) go to A, the rest to B.

```python
def two_city_sched_cost(costs):
    costs.sort(key=lambda c: c[0] - c[1])
    n = len(costs) // 2
    total = sum(a for a, _ in costs[:n]) + sum(b for _, b in costs[n:])
    return total
```

**Cost:** O(n log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every way to choose n people for city A. That costs O(C(2n, n) · n) time and O(n) space."

"I can do better: Sort by toA − toB. The first n (who save the most by going to A) go to A, the rest to B. That's O(n log n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/two-city-scheduling)

<a id="q18"></a>
### Q18. Remove Covered Intervals

🟡 Interview standard · *Sort by start asc, end desc*

Remove every interval covered by another (c ≤ a and b ≤ d). Return how many remain.

**Examples**

- `intervals = [[1, 4], [3, 6], [2, 8]]` → `2`
- `intervals = [[1, 4], [2, 3]]` → `1`

In [ ]:
def remove_covered_intervals(intervals):
    """
    Remove every interval covered by another (c ≤ a and b ≤ d). Return how many
    remain.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(remove_covered_intervals, [
    (([[1, 4], [3, 6], [2, 8]],), 2),
    (([[1, 4], [2, 3]],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each interval, check every other interval to see if it covers it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by start ascending and, for equal starts, end descending. An interval is covered exactly when its end ≤ the largest end seen so far.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each interval, check every other interval to see if it covers it.

```python
def remove_covered_intervals(intervals):
    n = len(intervals)
    return sum(1 for i in range(n) if not any(j != i and intervals[j][0] <= intervals[i][0] and intervals[i][1] <= intervals[j][1] for j in range(n)))
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort by start ascending and, for equal starts, end descending. An interval is covered exactly when its end ≤ the largest end seen so far.

```python
def remove_covered_intervals(intervals):
    intervals.sort(key=lambda x: (x[0], -x[1]))
    keep, far, kept = 0, 0, []
    for i, (s, e) in enumerate(intervals):
        if e > far:
            keep += 1
            far = e
            kept.append(i)
    return keep
```

**Cost:** O(n log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each interval, check every other interval to see if it covers it. That costs O(n²) time and O(1) space."

"I can do better: Sort by start ascending and, for equal starts, end descending. An interval is covered exactly when its end ≤ the largest end seen so far. That's O(n log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `intervals = [[1, 4], [3, 6], [2, 8]]`

1. [1,4] kept
2. [2,8] kept
3. [3,6] covered (end ≤ 8)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/remove-covered-intervals)

<a id="q19"></a>
### Q19. Car Pooling

🟡 Interview standard · *Sweep line over stops*

`trips[i] = [passengers, from, to]`. The car has `capacity` seats and drives east only. Can it complete all trips?

**Examples**

- `trips = [[2, 1, 5], [3, 3, 7]], capacity = 4` → `False`
- `trips = [[2, 1, 5], [3, 3, 7]], capacity = 5` → `True`

In [ ]:
def car_pooling(trips, capacity):
    """
    trips[i] = [passengers, from, to]. The car has capacity seats and drives
    east only. Can it complete all trips?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(car_pooling, [
    (([[2, 1, 5], [3, 3, 7]], 4), False),
    (([[2, 1, 5], [3, 3, 7]], 5), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every kilometre, add up the passengers of every trip covering it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Events: +p at 'from', −p at 'to'. Sort events (drop-offs before pick-ups at the same point) and sweep the running load.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every kilometre, add up the passengers of every trip covering it.

```python
def car_pooling(trips, capacity):
    end = max(t for _, _, t in trips)
    for km in range(end):
        if sum(p for p, f, t in trips if f <= km < t) > capacity:
            return False
    return True
```

**Cost:** O(n · D) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Events: +p at 'from', −p at 'to'. Sort events (drop-offs before pick-ups at the same point) and sweep the running load.

```python
def car_pooling(trips, capacity):
    events = sorted([(f, p) for p, f, _ in trips] + [(t, -p) for p, _, t in trips])
    load = 0
    for i, (x, delta) in enumerate(events):
        load += delta
        if load > capacity:
            return False
    return True
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every kilometre, add up the passengers of every trip covering it. That costs O(n · D) time and O(1) space."

"I can do better: Events: +p at 'from', −p at 'to'. Sort events (drop-offs before pick-ups at the same point) and sweep the running load. That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `trips = [[2, 1, 5], [3, 3, 7], [1, 5, 8]], capacity = 6`

1. At 1: pick up 2 → load 2
2. At 3: pick up 3 → load 5
3. At 5: drop off 2 → load 3
4. At 5: pick up 1 → load 4
5. At 7: drop off 3 → load 1
6. At 8: drop off 1 → load 0

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/car-pooling)

<a id="q20"></a>
### Q20. Candy

🔴 Stretch · *Two passes*

Each child gets ≥ 1 candy, and a child with a higher rating than a neighbour gets more than that neighbour. Return the minimum total.

**Examples**

- `ratings = [1, 0, 2]` → `5`
- `ratings = [1, 2, 2]` → `4`

In [ ]:
def candy(ratings):
    """
    Each child gets ≥ 1 candy, and a child with a higher rating than a neighbour
    gets more than that neighbour. Return the minimum total.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(candy, [
    (([1, 0, 2],), 5),
    (([1, 2, 2],), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Start everyone at 1 and keep sweeping, fixing any violated neighbour, until nothing changes.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Left → right: if rating rises, candy = left + 1. Right → left: if rating rises toward the left, candy = max(candy, right + 1). Both rules then hold.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Start everyone at 1 and keep sweeping, fixing any violated neighbour, until nothing changes.

```python
def candy(ratings):
    n = len(ratings)
    c = [1] * n
    changed = True
    while changed:
        changed = False
        for i in range(n):
            if i > 0 and ratings[i] > ratings[i - 1] and c[i] <= c[i - 1]:
                c[i], changed = c[i - 1] + 1, True
            if i < n - 1 and ratings[i] > ratings[i + 1] and c[i] <= c[i + 1]:
                c[i], changed = c[i + 1] + 1, True
    return sum(c)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Left → right: if rating rises, candy = left + 1. Right → left: if rating rises toward the left, candy = max(candy, right + 1). Both rules then hold.

```python
def candy(ratings):
    n = len(ratings)
    c = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            c[i] = c[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            c[i] = max(c[i], c[i + 1] + 1)
    return sum(c)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Start everyone at 1 and keep sweeping, fixing any violated neighbour, until nothing changes. That costs O(n²) time and O(n) space."

"I can do better: Left → right: if rating rises, candy = left + 1. Right → left: if rating rises toward the left, candy = max(candy, right + 1). Both rules then hold. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/greedy/candy)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is a greedy algorithm?</b></summary>

An algorithm that makes the locally optimal choice at each step without reconsidering, hoping it leads to a global optimum. It works when the problem has the greedy-choice property and optimal substructure.

</details>

<details>
<summary><b>2. How do you prove a greedy algorithm is correct?</b></summary>

Usually by an exchange argument: show that any optimal solution can be transformed, step by step, into the greedy solution without becoming worse. Alternatively, show greedy 'stays ahead' of any other solution at every step.

</details>

<details>
<summary><b>3. Why does activity selection sort by end time, not start time?</b></summary>

Choosing the interval that finishes earliest leaves the maximum remaining time for others. Sorting by start can pick a long early interval that blocks many short ones.

</details>

<details>
<summary><b>4. Give an example where greedy fails.</b></summary>

Coin change with coins {1, 3, 4} and amount 6: greedy picks 4 + 1 + 1 (3 coins), but 3 + 3 (2 coins) is optimal. 0/1 knapsack by value/weight ratio also fails.

</details>

<details>
<summary><b>5. Fractional vs 0/1 knapsack?</b></summary>

Fractional knapsack allows taking part of an item, so greedy by value/weight ratio is optimal (O(n log n)). In 0/1 knapsack items are indivisible, greedy fails, and DP (O(n · W)) is needed.

</details>

<details>
<summary><b>6. How do you find the minimum number of meeting rooms?</b></summary>

Sort start and end times (or events) and sweep: +1 at a start, −1 at an end; the maximum running count is the answer. Equivalently, sort by start and keep a min-heap of end times.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: What greedy means, and when it fails
- [ ] I can explain: Proving a greedy: the exchange argument
- [ ] I can explain: Intervals: sort, then merge
- [ ] I can explain: Scheduling: sort by end time
- [ ] I can explain: Reach and running balance
- [ ] I can explain: Sweep lines and two passes

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Interval List Intersections
- [ ] [Q17](#q17) Two City Scheduling
- [ ] [Q18](#q18) Remove Covered Intervals
- [ ] [Q19](#q19) Car Pooling
- [ ] [Q20](#q20) Candy

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.